# Phase 3 — Blind-A submission (fusion + LoRA-ColBERT + K2[frozen-CE])

End-to-end D1 spine on the fresh-start `mcrs` stack, producing a CodaBench `prediction.json`.

ONE flag drives everything:
- `BLIND=False` (dev-confirm): runs the IDENTICAL stack on the dev split (`test`) and prints the
  official `score_official` nDCG. Cheap proof the ColBERT channel + train+dev K2 compose and score
  sanely — run this FIRST.
- `BLIND=True` (submit): swaps in `talkpl-ai/TalkPlayData-Challenge-Blind-A` (80 sessions), runs the
  Gemini responder, writes the zip. Spends a limited slot — only after a clean dev-confirm.

Stack = RRF fusion (7 base channels) + the LoRA fine-tuned ColBERT (PLAID, focused query routed) ->
K2 (LGBM, with the FROZEN cross-encoder score as a leak-free feature). NO chained K3 (it hurt dev).
Train==serve: K2 is retrained on train(+dev for blind) with the SAME per-channel routing as serve.

## 1. Drive + HF + Gemini auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)
try:                                   # only needed when RESPONDER=='gemini'
    os.environ['GEMINI_API_KEY']=os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY'); print('Gemini key ok')
except Exception as e: print('no GEMINI_API_KEY secret (only needed for the responder):', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate peft google-genai
!pip -q uninstall -y torchao   # transformers wants torchao>0.16 but Colab ships 0.10 and its check RAISES; we don't use it
import sys; sys.path.insert(0,'.')

## 3. Config — set `BLIND` here

In [ ]:
# ── submission mode ──
BLIND=False                # False = dev-confirm + print nDCG; True = Blind-A submit (spends a slot)
RESPONDER='stub'           # 'stub' (predicted_response='ok', read nDCG axis only) | 'gemini' (full composite)
SMOKE=0                    # >0 caps serve turns for a quick pipeline check (0 = all)
SEED=42

# ── splits ──
ORG='talkpl-ai'
TRAIN_SESSIONS=3000        # train sessions for K2 (dev-confirm). BLIND adds the full dev split on top.
DEV_SESSIONS=1000          # dev sessions: the serve set when BLIND=False; folded into K2 train when BLIND=True
BLIND_DATASET=f'{ORG}/TalkPlayData-Challenge-Blind-A'   # 80 sessions, NO gold

# ── retrieval ──
TOPK=500                   # fused-pool depth into the reranker
SUBMIT_K=20                # ids per submission row (official cap)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'

# ── ColBERT (the LoRA fine-tune from phase2_colbert_finetune) ──
COLBERT_OUT_DIR=f'{OUT}/colbert/music-colbert-v1'   # merged plain-ColBERT checkpoint
Q_LEN=96; D_LEN=300; BSIZE=128
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_BASE='music-colbert-v1-d%d'%D_LEN

# ── K2 (LGBM) + frozen-CE feature ──
NEG_CAP=150; EARLY_STOPPING=50
K2_PARAMS={'learning_rate':0.05,'num_leaves':15,'min_child_samples':30,'reg_lambda':5.0,'reg_alpha':1.0,
           'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':1}
CE_STACK=True
CE_MODEL='BAAI/bge-reranker-v2-m3'; CROSS_ENCODER_K=50; CE_MAX_DOC_TOKENS=480

import random as _r, numpy as _np
_r.seed(SEED); _np.random.seed(SEED)
try:
    import torch as _t; _t.manual_seed(SEED); _t.cuda.manual_seed_all(SEED)
except Exception: pass
print('MODE:', 'BLIND-A SUBMIT' if BLIND else 'DEV-CONFIRM', '| responder', RESPONDER, '| smoke', SMOKE)

## 4. Catalog + base channels + dense doc_mat

In [ ]:
import glob, os, pickle, hashlib, pandas as pd, numpy as np
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB)); assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names); CKNN_MODS={l:m for l,m in CONTENT_MODALITIES.items() if m in _avail}
te={l:TrackEmbeddings(tre.select_columns(['track_id',m]),modalities=[m]) for l,m in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')

model=SentenceTransformer(DENSE_MODEL,device='cuda')
# config-hashed cache key (model + enriched version + catalog size) — never silently load a stale matrix
_dm_sig=hashlib.md5(f'{DENSE_MODEL}|enriched={USE_ENRICHED}|{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}'.encode()).hexdigest()[:8]
DOC_MAT_NPY=f'{OUT}/dense_doc_mat_{_dm_sig}.npy'
if os.path.exists(DOC_MAT_NPY):
    doc_mat=np.load(DOC_MAT_NPY); print('loaded cached doc_mat', doc_mat.shape)
else:
    doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
    np.save(DOC_MAT_NPY, doc_mat); print('cached doc_mat ->', DOC_MAT_NPY)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
cooc=pickle.load(open(COOC_PKL,'rb')) if os.path.exists(COOC_PKL) else build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat))
if not os.path.exists(COOC_PKL): pickle.dump(cooc,open(COOC_PKL,'wb'))
cknn=[ContentKNNChannel(te[l],m,label=l) for l,m in CKNN_MODS.items()]
base_chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
            CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
qb_full=QueryBuilder()                    # base channels' serve query (full history)
qb_focused=QueryBuilder(recency_window=1) # ColBERT's focused query (train==serve)
print('base channels:', [c.label for c in base_chans])

## 5. ColBERT PLAID channel (focused-query routed)

In [ ]:
from pylate import models
from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
from mcrs.retrieval.colbert_channel import colbert_doc_text
import hashlib

assert os.path.isdir(COLBERT_OUT_DIR), f'ColBERT checkpoint missing at {COLBERT_OUT_DIR} — run phase2_colbert_finetune first'
ft=models.ColBERT(model_name_or_path=COLBERT_OUT_DIR, query_length=Q_LEN, document_length=D_LEN)
doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=True)
# checkpoint-keyed PLAID index name so a retrain can't reuse a stale index (mismatched embedding spaces)
_ck_sig=hashlib.md5('|'.join(f'{f}:{os.stat(os.path.join(rt,f)).st_size}:{int(os.stat(os.path.join(rt,f)).st_mtime)}'
                             for rt,_,fs in os.walk(COLBERT_OUT_DIR) for f in sorted(fs)).encode()).hexdigest()[:8]
retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, f'{FT_IDX_BASE}-{_ck_sig}', doc_fn, batch_size=BSIZE)

class PlaidColBERTChannel:
    """Fusion channel backed by the fine-tuned ColBERT PLAID index. query_key='colbert' so the harness /
    build_rerank_groups route it the focused query while base channels keep the full query."""
    label='colbert'; query_key='colbert'
    def __init__(self, model, retriever): self.model, self.retriever = model, retriever
    def batch_text_to_item_retrieval(self, queries, topk, batch_context=None, user_ids=None):
        if not queries: return []
        return colbert_retrieve(self.model, self.retriever, list(queries), topk, batch_size=BSIZE)

colbert=PlaidColBERTChannel(ft, retr)
chans=base_chans+[colbert]
fusion=RRFFusion(chans, k=60)
labels=[c.label for c in chans]
PCQ={'colbert': qb_focused}               # per-channel routing: ColBERT gets the focused query
print('fusion channels:', labels)

## 6. Resolve train + serve splits (BLIND flag)

In [ ]:
# K2 trains on train(+dev when blind); serve = dev (with gold) when confirming, else Blind-A (no gold).
train_rows=dsd['train'] if not TRAIN_SESSIONS else dsd['train'].select(range(min(TRAIN_SESSIONS,len(dsd['train']))))
conv_train_parts=[Conversations(train_rows)]
if BLIND:
    conv_train_parts.append(Conversations(dsd['test']))                      # fold the full dev split into K2 train
    blind_rows=load_dataset(BLIND_DATASET, split='test')
    conv_serve=Conversations(blind_rows); has_gold=False
    print('BLIND: train = train +', len(dsd['test']), 'dev sessions | serve = Blind-A', len(blind_rows), 'sessions')
else:
    conv_serve=Conversations(dsd['test'].select(range(min(DEV_SESSIONS,len(dsd['test']))))); has_gold=True
    print('DEV-CONFIRM: train =', len(train_rows), 'sessions | serve =', DEV_SESSIONS, 'dev sessions')

train_turns=[t for cv in conv_train_parts for t in cv.turns()]
serve_turns=list(conv_serve.turns())
if SMOKE: serve_turns=serve_turns[:SMOKE]; print('SMOKE: capped serve to', len(serve_turns), 'turns')
def train_gold(t):
    for cv in conv_train_parts:
        g=cv.gold(t.session_id,t.turn_number)
        if g is not None: return g
    return None
print('train turns', len(train_turns), '| serve turns', len(serve_turns))

## 7. Retrain K2 (routed pools + frozen-CE feature, train==serve)

In [ ]:
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.rerank.train import build_rerank_groups
from mcrs.rerank.neural import NeuralReranker, build_ce_score_lookup, make_ce_feature_fn
from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn

# dense query<->candidate cosine as a K2 feature; precompute query vecs for ALL turns (train+serve)
_qcache={}
def precompute_qvecs(turns):
    mat=model.encode([DENSE_QUERY_PREFIX+qb_full.build(t).text for t in turns],batch_size=256,normalize_embeddings=True)
    _qcache.update({(t.session_id,t.turn_number): mat[i] for i,t in enumerate(turns)})
def dense_cos(ctx, tid):
    j=cat.id_to_index.get(tid)
    if j is None: return 0.0
    v=_qcache.get((ctx.session_id,ctx.turn_number))
    if v is None:
        v=model.encode([DENSE_QUERY_PREFIX+qb_full.build(ctx).text],normalize_embeddings=True)[0]; _qcache[(ctx.session_id,ctx.turn_number)]=v
    return float(v @ doc_mat[j])
precompute_qvecs(train_turns); precompute_qvecs(serve_turns)

# routed train pools (ColBERT focused) + routed serve pools (reused for CE scoring)
print('building K2 train groups (runs fusion incl ColBERT over', len(train_turns), 'turns)...')
groups=build_rerank_groups(qb_full, fusion, train_turns, train_gold, topk=TOPK, per_channel_query_builders=PCQ)
serve_q=[qb_full.build(t).text for t in serve_turns]
serve_bc=[{'history_tids':t.history_tids,'user_id':t.user_id} for t in serve_turns]
serve_pcq={'colbert':[qb_focused.build(t).text for t in serve_turns]}
serve_pools=fusion.fuse(serve_q, TOPK, batch_context=serve_bc, user_ids=[t.user_id for t in serve_turns],
                        per_channel_queries=serve_pcq)

score_fns={'dense_cos': dense_cos}
if CE_STACK:
    # FROZEN cross-encoder score as a leak-free K2 feature. Score it over the TRAIN pools AND the SERVE
    # pools (else serve candidates hit the -1.0 sentinel and the CE signal silently vanishes at serve).
    _frozen_ce=build_cross_encoder_score_fn(CE_MODEL,device='cuda',max_length=512,max_doc_tokens=CE_MAX_DOC_TOKENS,dtype='fp16')
    _nr=NeuralReranker(cat,qb_full,_frozen_ce,cross_encoder_k=CROSS_ENCODER_K,enriched=USE_ENRICHED)
    ce_lookup={}
    print('scoring frozen CE over train pools (slow)...')
    ce_lookup.update(build_ce_score_lookup([g[0] for g in groups],[g[1] for g in groups],_nr,normalize=True))
    print('scoring frozen CE over serve pools...')
    ce_lookup.update(build_ce_score_lookup(serve_turns, serve_pools, _nr, normalize=True))
    score_fns['ce_score']=make_ce_feature_fn(ce_lookup, default=-1.0)
    print('ce_lookup entries:', len(ce_lookup))

fb=FeatureBuilder(cat, labels, score_fns=score_fns)
k2=LGBMReranker(fb,n_estimators=500,params=K2_PARAMS,neg_cap=NEG_CAP,
                early_stopping_rounds=EARLY_STOPPING,val_fraction=0.1).fit(groups)
_tag='blind' if BLIND else 'dev'
k2.save(f'{OUT}/k2_lgbm_{_tag}.txt')
print(f'train_groups={k2.n_train_groups_} val_groups={k2.n_val_groups_} -> saved {OUT}/k2_lgbm_{_tag}.txt')
print('top features:',sorted(zip(fb.feature_names,k2.model.feature_importances_),key=lambda x:-x[1])[:8])

## 8. Inference (routed serve) -> rows (+ nDCG when dev-confirm)

In [ ]:
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness, validate_submission

harness=InferenceHarness(qb_full, fusion, TopKAssembler(cat, top_k=SUBMIT_K), reranker=k2,
                         topk=TOPK, per_channel_query_builders=PCQ)
rows=harness.run(serve_turns, show_progress=True)
validate_submission(rows, catalog=cat, expected_keys=[(t.session_id,t.turn_number) for t in serve_turns])
print('rows:', len(rows))

if has_gold:
    from mcrs.eval.harness import GoldRow
    from mcrs.eval.official import score_official
    golds=[GoldRow(t.session_id,t.user_id,t.turn_number,conv_serve.gold(t.session_id,t.turn_number))
           for t in serve_turns if conv_serve.gold(t.session_id,t.turn_number) is not None]
    rows_g=[r for r in rows if (r.session_id,r.turn_number) in {(g.session_id,g.turn_number) for g in golds}]
    sc=score_official(rows_g, golds, catalog_size=len(cat.index_to_id))
    print('DEV-CONFIRM official:', {k:round(v,4) for k,v in sc.items() if 'ndcg' in k or 'diversity' in k})
    print('>>> read ndcg@20; if sane, set BLIND=True and re-run from cell 6 to submit.')

## 9. Responder + package `prediction.json`

In [ ]:
import json, zipfile, datetime, subprocess
from mcrs.run.harness import write_submission

_tag='blind' if BLIND else 'dev'
BASE=f'{OUT}/prediction_{_tag}_base.json'
write_submission(rows, BASE)
print('wrote base prediction ->', BASE, '| rows', len(rows))

FINAL=BASE
if RESPONDER=='gemini':
    DSET=BLIND_DATASET if BLIND else f'{ORG}/TalkPlayData-Challenge-Dataset'
    FINAL=f'{OUT}/prediction_{_tag}_gemini.json'
    subprocess.run(['python','-u','salvage/scripts/gemini_responder.py','--pred',BASE,'--out',FINAL,
                    '--dataset',DSET,'--top-n','3','--best-of','1','--judge-model','gemini-2.5-flash','--sleep','0.2'],
                   check=True)
    assert os.path.exists(FINAL), 'gemini responder did not write output — read the traceback'
else:
    print("RESPONDER='stub': predicted_response stays 'ok' (nDCG axis only; LLM axis ~1/5 by design)")

preds=json.load(open(FINAL))
if BLIND:
    assert len(preds)==80, f'expected 80 Blind-A rows, got {len(preds)} — DO NOT submit'
ZIP=f'{DRIVE}/recsys2026_submissions/{datetime.date.today().isoformat()}-{_tag}-fusion-colbert-k2ce.zip'
os.makedirs(os.path.dirname(ZIP),exist_ok=True)
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(FINAL, arcname='prediction.json')   # MUST be 'prediction.json' at zip ROOT for CodaBench
print('submission zip ready:', ZIP)
print('UPLOAD to CodaBench' if BLIND else 'DEV zip (not for upload — flip BLIND=True to submit)')